## My Capstone Plan

**Domain:** Study Buddy — B.Tech Physics

**User:** B.Tech students (1st and 2nd year) who need help understanding Physics concepts,
laws, and formulas from the course syllabus at any hour — especially before exams or late
at night when no professor is available.

**Success looks like:** The agent correctly answers ≥ 90% of B.Tech-level physics questions
using only the knowledge base (no hallucinated formulas), correctly routes calculator
questions to the tool, remembers context within a session, and admits when a topic is
outside its knowledge base.

**Tool I will add:** A safe scientific calculator — evaluates mathematical expressions
including arithmetic, trigonometry (sin/cos/tan in degrees), sqrt, log, and all key physics
constants (G, h, c, g, k, R, NA, mu0, eps0). Essential because students need to substitute
numbers into formulas and the knowledge base can explain but cannot compute.

**Deployment choice:** Streamlit UI — students open it in a browser during study sessions.

---
## 0. Setup

In [1]:
# ============================================================
# LOCAL SETUP 
# ============================================================
# pip install -r requirements.txt
#
# Create a .env file in the project root with:
#   GROQ_API_KEY=your_key_here

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key : {'OK — loaded' if len(groq_key) > 10 else 'MISSING — add to .env'}")
print(f"LangGraph    : {version('langgraph')}")

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
r = llm.invoke("Say ready in 1 word.")
print(f"LLM          : {r.content}")

c:\Users\KIIT0001\OneDrive\Desktop\Agentic_AI\Capstone project\.capstone\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Groq API Key : OK — loaded
LangGraph    : 0.3.21
LLM          : Ready.


---
## Part 1 — Domain Setup: Knowledge Base

12 Physics documents — one topic per document, 150-400 words each.

**Design decision:** Each document covers exactly ONE topic. This maximises retrieval
precision — a vague multi-topic document embeds poorly and returns irrelevant chunks.

In [3]:
from agent import DOCUMENTS, build_knowledge_base

print("Loading embedding model (downloads ~90 MB on first run)...")
embedder, collection = build_knowledge_base()

print(f"\nKnowledge base ready: {collection.count()} documents")
for d in DOCUMENTS:
    print(f"   [{d['id']}] {d['topic']}")

Loading embedding model (downloads ~90 MB on first run)...
Loading embedding model (all-MiniLM-L6-v2) — ~90 MB on first run...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given



Knowledge base ready: 12 documents
   [doc_001] Newton's Laws of Motion
   [doc_002] Work, Energy, and Power
   [doc_003] Circular Motion and Gravitation
   [doc_004] Waves and Simple Harmonic Motion
   [doc_005] Thermodynamics — Laws and Processes
   [doc_006] Electrostatics — Coulomb's Law and Electric Field
   [doc_007] Current Electricity — Ohm's Law and Circuits
   [doc_008] Magnetic Fields and Electromagnetic Induction
   [doc_009] Optics — Ray Optics and Wave Optics
   [doc_010] Modern Physics — Quantum Theory and Photoelectric Effect
   [doc_011] Special Relativity
   [doc_012] Fluid Mechanics and Surface Tension


In [4]:
test_queries = [
    "What is Newton's second law and what is the formula?",
    "How does the photoelectric effect work?",
    "What is Bernoulli's equation and when do we use it?",
]

for q in test_queries:
    q_emb   = embedder.encode([q]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=2)
    print(f"Query: {q}")
    for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
        print(f"  [{i+1}] {meta['topic']}")
        print(f"       {doc[:130]}...")
    print()

print("If the retrieved chunks are relevant — retrieval is working correctly.")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: What is Newton's second law and what is the formula?
  [1] Newton's Laws of Motion
       Newton's Three Laws of Motion form the foundation of classical mechanics.

First Law (Law of Inertia): An object remains at rest o...
  [2] Circular Motion and Gravitation
       Circular Motion: Object moves along a circular path. In uniform circular motion,
speed is constant but velocity direction changes ...

Query: How does the photoelectric effect work?
  [1] Modern Physics — Quantum Theory and Photoelectric Effect
       Modern Physics: phenomena beyond classical mechanics.

Planck's Hypothesis: energy emitted/absorbed in discrete quanta. E = h*f = ...
  [2] Optics — Ray Optics and Wave Optics
       Optics: behaviour and properties of light.

Reflection: angle of incidence = angle of reflection (both measured from normal).

Sne...

Query: What is Bernoulli's equation and when do we use it?
  [1] Fluid Mechanics and Surface Tension
       Fluid Mechanics: behaviour of liquids and gase

---
## Part 2 — State Design

Every field a node writes must appear here — missing fields cause KeyError at runtime.

In [5]:
from agent import CapstoneState

print("CapstoneState fields:", list(CapstoneState.__annotations__.keys()))

# ── Field-by-field explanation ─────────────────────────────────────────────────
# question      : current user input
# messages      : sliding window of last 3 turns (6 message dicts)
# route         : "retrieve" | "memory_only" | "tool" — set by router_node
# retrieved     : formatted ChromaDB context string — set by retrieval_node
# sources       : list of topic names of retrieved chunks
# tool_result   : output of the scientific calculator — set by tool_node
# answer        : final LLM response — set by answer_node
# faithfulness  : score 0.0-1.0 — set by eval_node
# eval_retries  : retry counter (safety valve, max=2) — set by eval_node
# student_name  : extracted from "my name is X" in conversation
# quiz_score    : optional running self-quiz score

CapstoneState fields: ['question', 'messages', 'route', 'retrieved', 'sources', 'tool_result', 'answer', 'faithfulness', 'eval_retries', 'student_name', 'quiz_score']


---
## Part 3 — Node Functions

Each node is tested **in isolation** before graph assembly.

In [6]:
# ── Node 1: Memory ────────────────────────────────────────────────────────────

from agent import make_memory_node

memory_node = make_memory_node()

# ── Isolation tests ────────────────────────────────────────────────────────────

state = {"question": "My name is Siddharth. What is Newton's second law?", "messages": [], "student_name": ""}
result = memory_node(state)
print(f"messages     : {result['messages']}")
print(f"student_name : {result['student_name']}")
print("memory_node OK")

# Sliding window: 6 existing msgs + 1 new = 7, should trim to 6

msgs_6 = [{"role": "user", "content": f"q{i}"} for i in range(6)]
r2 = memory_node({"question": "new question", "messages": msgs_6, "student_name": ""})
assert len(r2["messages"]) == 6, f"Sliding window failed: {len(r2['messages'])}"
print(f"sliding window: 7 msgs trimmed to {len(r2['messages'])} — OK")

messages     : [{'role': 'user', 'content': "My name is Siddharth. What is Newton's second law?"}]
student_name : Siddharth
memory_node OK
sliding window: 7 msgs trimmed to 6 — OK


In [7]:
# ── Node 2: Router ────────────────────────────────────────────────────────────

from agent import make_router_node

router_node = make_router_node(llm)

# ── Isolation tests ────────────────────────────────────────────────────────────

tests = [
    ("Explain Newton's second law.",              "retrieve"),
    ("What did you just say?",                    "memory_only"),
    ("calculate sqrt(2 * 9.8 * 10)",              "tool"),
    ("What is the period of a simple pendulum?",  "retrieve"),
    ("Can you repeat that?",                      "memory_only"),
]
for q, expected in tests:
    r = router_node({"question": q, "messages": []})
    status = "OK" if r["route"] == expected else f"UNEXPECTED (got {r['route']})"
    print(f"[{status}] '{q[:55]}' -> {r['route']}")

  [router] -> retrieve
[OK] 'Explain Newton's second law.' -> retrieve
  [router] -> memory_only
[OK] 'What did you just say?' -> memory_only
  [router] -> tool
[OK] 'calculate sqrt(2 * 9.8 * 10)' -> tool
  [router] -> retrieve
[OK] 'What is the period of a simple pendulum?' -> retrieve
  [router] -> memory_only
[OK] 'Can you repeat that?' -> memory_only


In [8]:
# ── Node 3: Retrieval ─────────────────────────────────────────────────────────

from agent import make_retrieval_node, skip_retrieval_node

retrieval_node = make_retrieval_node(embedder, collection)

# ── Isolation test ─────────────────────────────────────────────────────────────

test_state = {"question": "What is the formula for centripetal force?"}
result = retrieval_node(test_state)
print(f"sources  : {result['sources']}")
print(f"context  : {result['retrieved'][:200]}...")
print("retrieval_node OK")
print()

skip_result = skip_retrieval_node({})
print(f"skip_retrieval_node: retrieved='{skip_result['retrieved']}', sources={skip_result['sources']}")
print("skip_retrieval_node OK")

sources  : ['Circular Motion and Gravitation', "Newton's Laws of Motion", 'Work, Energy, and Power']
context  : [Circular Motion and Gravitation]
Circular Motion: Object moves along a circular path. In uniform circular motion,
speed is constant but velocity direction changes — so acceleration exists.

Centripet...
retrieval_node OK

skip_retrieval_node: retrieved='', sources=[]
skip_retrieval_node OK


In [9]:
# ── Node 4: Tool — Scientific Calculator ──────────────────────────────────────
# Tool chosen: safe scientific calculator with physics constants.
# Handles: arithmetic, trig (degrees), sqrt, log, ln, exp
# Constants: G, h, c, g, k, R, NA, mu0, eps0, pi, e

from agent import tool_node

# ── Isolation tests ────────────────────────────────────────────────────────────

tests = [
    "calculate sqrt(2 * 9.8 * 10)",          
    "calculate 9e9 * 1e-6 * 2e-6 / 0.1**2", 
    "calculate sin(30)",                      
    "what is the colour of the sky",          
]
for q in tests:
    result = tool_node({"question": q})
    print(f"Q: {q}")
    print(f"   {result['tool_result'][:110]}")
    print()
print("tool_node OK — note: graceful fail returns error string, does not raise exception")

  [tool] Calculator result: sqrt(2 * 9.8 * 10) = 14
(Constants available: G=6.674e-11, h=6.626e-34, c=3000000
Q: calculate sqrt(2 * 9.8 * 10)
   Calculator result: sqrt(2 * 9.8 * 10) = 14
(Constants available: G=6.674e-11, h=6.626e-34, c=300000000.0, g=9.

  [tool] Calculator result: 9e9 * 1e-6 * 2e-6 / 0.1**2 = 1.8
(Constants available: G=6.674e-11, h=6.626e-34, 
Q: calculate 9e9 * 1e-6 * 2e-6 / 0.1**2
   Calculator result: 9e9 * 1e-6 * 2e-6 / 0.1**2 = 1.8
(Constants available: G=6.674e-11, h=6.626e-34, c=30000000

  [tool] Calculator result: sin(30) = 0.5
(Constants available: G=6.674e-11, h=6.626e-34, c=300000000.0, g=9.
Q: calculate sin(30)
   Calculator result: sin(30) = 0.5
(Constants available: G=6.674e-11, h=6.626e-34, c=300000000.0, g=9.8, k=90000

  [tool] Calculator could not evaluate that expression. Please write it clearly, e.g. 'calculate sqrt(2*g*10)
Q: what is the colour of the sky
   Calculator could not evaluate that expression. Please write it clearly, e.g. 'calculat

In [10]:
# ── Node 5: Answer ────────────────────────────────────────────────────────────

from agent import make_answer_node

answer_node = make_answer_node(llm)

# ── Isolation test ─────────────────────────────────────────────────────────────

test_state = {
    "question"    : "What is the period formula for a simple pendulum?",
    "retrieved"   : (
        "[Waves and Simple Harmonic Motion]\n"
        "Simple Pendulum: T = 2*pi * sqrt(L/g) — period depends only on L and g, NOT mass."
    ),
    "tool_result" : "",
    "messages"    : [],
    "eval_retries": 0,
    "student_name": "Siddharth",
}
result = answer_node(test_state)
print(f"answer: {result['answer'][:300]}")
print("answer_node OK")

answer: Siddharth, the period formula for a simple pendulum is: 

T = 2 * π * sqrt(L/g)

where:
- T = period in seconds (s)
- π (pi) is a mathematical constant approximately equal to 3.14159
- L = length of the pendulum in meters (m)
- g = acceleration due to gravity in meters per second squared (m/s^2)
- s
answer_node OK


In [11]:
# ── Node 6: Eval  &  Node 7: Save ─────────────────────────────────────────────

from agent import make_eval_node, make_save_node, FAITHFULNESS_THRESHOLD, MAX_EVAL_RETRIES

eval_node = make_eval_node(llm)
save_node = make_save_node()

print(f"Faithfulness threshold : {FAITHFULNESS_THRESHOLD}")
print(f"Max eval retries       : {MAX_EVAL_RETRIES}")

# ── eval_node isolation test ───────────────────────────────────────────────────

eval_state = {
    "answer"      : "The period is T = 2*pi*sqrt(L/g) where L is length and g is gravity.",
    "retrieved"   : "Simple Pendulum: T = 2*pi * sqrt(L/g) — period depends only on L and g, NOT mass.",
    "eval_retries": 0,
}
r = eval_node(eval_state)
print(f"\neval_node test: faithfulness={r['faithfulness']:.2f}, retries={r['eval_retries']}")

# ── save_node isolation test ───────────────────────────────────────────────────

save_state = {
    "messages": [{"role": "user", "content": "What is SHM?"}],
    "answer"  : "SHM is periodic motion where F = -kx.",
}
r = save_node(save_state)
print(f"save_node test: messages={r['messages']}")
print("eval_node and save_node OK")

Faithfulness threshold : 0.7
Max eval retries       : 2
  [eval] faithfulness=1.00 -> PASS

eval_node test: faithfulness=1.00, retries=1
save_node test: messages=[{'role': 'user', 'content': 'What is SHM?'}, {'role': 'assistant', 'content': 'SHM is periodic motion where F = -kx.'}]
eval_node and save_node OK


---
## Part 4 — Graph Assembly

Architecture: `memory -> router -> [retrieve | skip | tool] -> answer -> eval -> save -> END`

Two conditional edges:
1. After `router`: `route_decision` maps `state.route` to the correct branch
2. After `eval`: `eval_decision` retries answer or proceeds to save

In [12]:
# ── Routing functions and graph builder ───────────────────────────────────────

from agent import route_decision, eval_decision
from agent import make_memory_node, make_router_node, make_retrieval_node, skip_retrieval_node
from agent import tool_node, make_answer_node, make_eval_node, make_save_node
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

assert route_decision({"route": "tool"})        == "tool"
assert route_decision({"route": "memory_only"}) == "skip"
assert route_decision({"route": "retrieve"})    == "retrieve"
assert route_decision({})                        == "retrieve"   
print("route_decision: all 4 cases correct")

assert eval_decision({"faithfulness": 0.9, "eval_retries": 1}) == "save"    # above threshold
assert eval_decision({"faithfulness": 0.5, "eval_retries": 1}) == "answer"  # below, retry
assert eval_decision({"faithfulness": 0.4, "eval_retries": 2}) == "save"    # max retries hit
print("eval_decision: all 3 cases correct")

# Local graph builder: same structure as agent.build_graph, but with a branch
# label that does not collide with the state key "answer" in LangGraph.
def build_graph(llm, embedder, collection):
    memory_node    = make_memory_node()
    router_node    = make_router_node(llm)
    retrieval_node = make_retrieval_node(embedder, collection)
    answer_node    = make_answer_node(llm)
    eval_node      = make_eval_node(llm)
    save_node      = make_save_node()

    graph = StateGraph(CapstoneState)

    graph.add_node("memory",   memory_node)
    graph.add_node("router",   router_node)
    graph.add_node("retrieve", retrieval_node)
    graph.add_node("skip",     skip_retrieval_node)
    graph.add_node("tool",     tool_node)
    graph.add_node("generate_answer", answer_node) 
    graph.add_node("eval",     eval_node)
    graph.add_node("save",     save_node)

    graph.set_entry_point("memory")
    graph.add_edge("memory",   "router")

    graph.add_edge("retrieve", "generate_answer")
    graph.add_edge("skip",     "generate_answer")
    graph.add_edge("tool",     "generate_answer")
    graph.add_edge("generate_answer", "eval")
    graph.add_edge("save",     END)

    def eval_route(state):
        return "generate_answer" if eval_decision(state) == "answer" else "save"

    graph.add_conditional_edges(
        "router", route_decision,
        {"retrieve": "retrieve", "skip": "skip", "tool": "tool"},
    )

    graph.add_conditional_edges(
        "eval", eval_route,
        {"generate_answer": "generate_answer", "save": "save"},
    )

    app = graph.compile(checkpointer=MemorySaver())
    return app

app = build_graph(llm, embedder, collection)
print("\nGraph compiled successfully!")
print("Nodes: memory -> router -> [retrieve|skip|tool] -> answer -> eval -> save -> END")


route_decision: all 4 cases correct
eval_decision: all 3 cases correct

Graph compiled successfully!
Nodes: memory -> router -> [retrieve|skip|tool] -> answer -> eval -> save -> END


---
## Part 5 — Testing

10 questions: 8 domain questions (including 1 multi-turn memory test) + 2 red-team tests.

Red-team categories:
- **Out-of-scope**: topic not in KB → agent must admit it doesn't know
- **False premise**: factually wrong assumption → agent must correct it

In [14]:
def ask(question: str, thread_id: str = "test") -> dict:
    "Helper: invoke the compiled agent and return the full result dict."
    config = {"configurable": {"thread_id": thread_id}}
    return app.invoke({"question": question, "student_name": ""}, config=config)

r = ask("What formula gives the period of a simple pendulum?", thread_id="smoke")
print(f"Route   : {r.get('route')}")
print(f"Sources : {r.get('sources')}")
print(f"Faith   : {r.get('faithfulness', 0):.2f}")
print(f"Answer  : {r.get('answer', '')[:200]}")

  [router] -> retrieve
  [eval] faithfulness=1.00 -> PASS
Route   : retrieve
Sources : ['Waves and Simple Harmonic Motion', 'Circular Motion and Gravitation', 'Work, Energy, and Power']
Faith   : 1.00
Answer  : The formula for the period of a simple pendulum is: 

T = 2*pi * sqrt(L/g)

where T = period, L = length of the pendulum, g = acceleration due to gravity (approximately 9.8 m/s²).


In [15]:
TEST_QUESTIONS = [
    {
        "q"       : "Explain Newton's second law and give the formula with units.",
        "expect"  : "F = ma, units explained (N, kg, m/s^2)",
        "red_team": False,
    },
    {
        "q"       : "What is the work-energy theorem?",
        "expect"  : "W_net = delta_KE = half*m*v_f^2 - half*m*v_i^2",
        "red_team": False,
    },
    {
        "q"       : "What is SHM and what is the period of a simple pendulum?",
        "expect"  : "T = 2*pi*sqrt(L/g); period independent of mass",
        "red_team": False,
    },
    {
        "q"       : "State the first law of thermodynamics and explain each term.",
        "expect"  : "delta_U = Q - W; sign convention explained",
        "red_team": False,
    },
    {
        "q"       : "What is Coulomb's law? Write the formula and define every symbol.",
        "expect"  : "F = k*q1*q2/r^2; k, q, r defined with units",
        "red_team": False,
    },
    {
        "q"       : "What is Faraday's law and what does Lenz's law say?",
        "expect"  : "EMF = -d(Phi_B)/dt; Lenz opposes change in flux",
        "red_team": False,
    },
    {
        "q"       : "Write Snell's law and explain total internal reflection.",
        "expect"  : "n1*sin(t1) = n2*sin(t2); critical angle, dense to rare medium",
        "red_team": False,
    },
    {
        "q"       : "You just told me Coulomb's law. Now calculate the force between two charges of 1 uC each separated by 10 cm.",
        "expect"  : "Route to tool; F = k * 1e-6 * 1e-6 / 0.1^2 = 0.9 N",
        "red_team": False,
    },
    {
        "q"       : "Explain string theory and the 11 dimensions it predicts in detail.",
        "expect"  : "Should admit this is not in the knowledge base",
        "red_team": True,
    },
    {
        "q"       : "I heard that the speed of light is different in different inertial frames according to Einstein. Can you confirm?",
        "expect"  : "Should CORRECT the false premise — Einstein's 2nd postulate says c is CONSTANT",
        "red_team": True,
    },
]

print(f"Test suite: {len(TEST_QUESTIONS)} questions "
      f"({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")

test_results = []

print("=" * 65)
print("RUNNING TEST SUITE — Physics Study Buddy")
print("=" * 65)

for i, test in enumerate(TEST_QUESTIONS):
    label = "[RED-TEAM]" if test["red_team"] else f"[Test {i+1:02d}]"
    print(f"\n{label} {test['q'][:80]}")

    result = ask(test["q"], thread_id=f"test-{i}")
    answer = result.get("answer", "")
    faith  = result.get("faithfulness", 0.0)
    route  = result.get("route", "?")

    print(f"  Route    : {route}")
    print(f"  Faith    : {faith:.2f}")
    print(f"  Answer   : {answer[:180]}")
    print(f"  Expected : {test['expect']}")

    if test["red_team"]:
        pass_signals = [
            "don't have", "not in my knowledge", "not in the knowledge",
            "textbook", "professor", "constant for all", "same for all",
            "incorrect", "not correct", "actually", "contrary",
            "second postulate", "same in all inertial",
        ]
        passed = any(sig.lower() in answer.lower() for sig in pass_signals)
    else:
        passed = len(answer) > 50 and faith >= 0.6

    print(f"  Result   : {'PASS' if passed else 'FAIL'}")
    test_results.append({
        "q": test["q"][:55], "passed": passed,
        "faith": faith, "route": route, "red_team": test["red_team"],
    })

total  = len(test_results)
n_pass = sum(1 for r in test_results if r["passed"])
red_p  = sum(1 for r in test_results if r["red_team"] and r["passed"])
avg_f  = sum(r["faith"] for r in test_results) / total

print(f"\n{'='*65}")
print(f"RESULTS  : {n_pass}/{total} passed")
print(f"Red-team : {red_p}/2 passed")
print(f"Avg faithfulness : {avg_f:.2f}")

Test suite: 10 questions (2 red-team)
RUNNING TEST SUITE — Physics Study Buddy

[Test 01] Explain Newton's second law and give the formula with units.
  [router] -> retrieve
  [eval] faithfulness=1.00 -> PASS
  Route    : retrieve
  Faith    : 1.00
  Answer   : Newton's Second Law, also known as the Law of Acceleration, states that the net force acting on an object is equal to the mass of the object multiplied by its acceleration. 

The f
  Expected : F = ma, units explained (N, kg, m/s^2)
  Result   : PASS

[Test 02] What is the work-energy theorem?
  [router] -> retrieve
  [eval] faithfulness=0.00 -> RETRY
  [eval] faithfulness=0.00 -> RETRY
  Route    : retrieve
  Faith    : 0.00
  Answer   : The work-energy theorem is given by the formula: 
W_net = delta_KE = (1/2) * m * v_f^2 - (1/2) * m * v_i^2, 
where 
- W_net = net work done (in Joules, J),
- delta_KE = change in k
  Expected : W_net = delta_KE = half*m*v_f^2 - half*m*v_i^2
  Result   : FAIL

[Test 03] What is SHM and what is t

---
## Part 6 — RAGAS Baseline Evaluation

In [16]:
RAGAS_QUESTIONS = [
    {
        "question"    : "What is Newton's second law?",
        "ground_truth": (
            "Newton's second law states that the net force on an object equals "
            "its mass times acceleration: F = ma. Force is in Newtons (N = kg·m/s²)."
        ),
    },
    {
        "question"    : "What is the period of a simple pendulum?",
        "ground_truth": (
            "The period of a simple pendulum is T = 2*pi*sqrt(L/g), where L is "
            "the length and g = 9.8 m/s². The period does not depend on mass."
        ),
    },
    {
        "question"    : "State the first law of thermodynamics.",
        "ground_truth": (
            "The first law of thermodynamics is delta_U = Q - W, where delta_U is "
            "change in internal energy, Q is heat added to the system, and W is work "
            "done by the system. Q > 0 if heat flows in; W > 0 if system expands."
        ),
    },
    {
        "question"    : "What is Coulomb's law?",
        "ground_truth": (
            "Coulomb's law: F = k*q1*q2/r^2, where k = 9e9 N·m²/C², q1 and q2 are "
            "the charges in Coulombs, and r is the distance between them in meters."
        ),
    },
    {
        "question"    : "What does E = mc^2 mean?",
        "ground_truth": (
            "E = mc^2 is mass-energy equivalence from special relativity: the rest "
            "energy of an object equals its mass times the square of the speed of light. "
            "Total energy is E = gamma*m*c^2 and relativistic KE = (gamma-1)*m*c^2."
        ),
    },
]

eval_dataset = []
print("Running agent for RAGAS evaluation dataset...")

for rq in RAGAS_QUESTIONS:
    q_emb   = embedder.encode([rq["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    result  = ask(rq["question"], thread_id=f"ragas-{rq['question'][:12]}")
    eval_dataset.append({
        "question"    : rq["question"],
        "answer"      : result.get("answer", ""),
        "contexts"    : chunks,
        "ground_truth": rq["ground_truth"],
    })
    print(f"  done: {rq['question'][:60]}")

print(f"\nEval dataset built: {len(eval_dataset)} rows")

Running agent for RAGAS evaluation dataset...
  [router] -> retrieve
  [eval] faithfulness=1.00 -> PASS
  done: What is Newton's second law?
  [router] -> retrieve
  [eval] faithfulness=1.00 -> PASS
  done: What is the period of a simple pendulum?
  [router] -> retrieve
  [eval] faithfulness=0.50 -> RETRY
  [eval] faithfulness=1.00 -> PASS
  done: State the first law of thermodynamics.
  [router] -> retrieve
  [eval] faithfulness=0.50 -> RETRY
  [eval] faithfulness=1.00 -> PASS
  done: What is Coulomb's law?
  [router] -> retrieve
  [eval] faithfulness=0.00 -> RETRY
  [eval] faithfulness=0.00 -> RETRY
  done: What does E = mc^2 mean?

Eval dataset built: 5 rows


In [ ]:
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    ragas_data = Dataset.from_list(eval_dataset)
    print("Running RAGAS evaluation (1-2 minutes)...")

    ragas_result = evaluate(
        dataset=ragas_data,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    df = ragas_result.to_pandas()
    print("\n" + "=" * 45)
    print("BASELINE RAGAS SCORES")
    print("=" * 45)
    print(f"Faithfulness:      {df['faithfulness'].mean():.3f}")
    print(f"Answer Relevance:  {df['answer_relevancy'].mean():.3f}")
    print(f"Context Precision: {df['context_precision'].mean():.3f}")
    print("\nRecord these baseline scores. Re-run after any improvements.")

except ImportError:
    print("RAGAS not installed — running manual faithfulness scoring")
    faith_scores = []
    for row in eval_dataset:
        prompt = f"""Rate faithfulness 0.0-1.0. Reply with only a number.
Context: {row['contexts'][0][:300]}
Answer: {row['answer'][:200]}"""
        try:
            score = float(llm.invoke(prompt).content.strip().split()[0])
            score = max(0.0, min(1.0, score))
        except Exception:
            score = 0.5
        faith_scores.append(score)
        print(f"  Q: {row['question'][:45]:45s} -> {score:.2f}")

    avg = sum(faith_scores) / len(faith_scores)
    print(f"\nBaseline faithfulness: {avg:.3f}")
    print("Install RAGAS for full evaluation: pip install ragas datasets")

---
## Part 7 — Deployment

In [18]:
# ── Part 7: Deployment ────────────────────────────────────────────────────────

import subprocess, sys

result = subprocess.run(
    [sys.executable, "-c", "import capstone_streamlit; print('Import OK')"],
    capture_output=True, text=True,
)
if "OK" in result.stdout:
    print("capstone_streamlit.py imports successfully.")
    print("Run:  streamlit run capstone_streamlit.py")
else:
    print("Syntax/import error:")
    print(result.stderr[:400] if result.stderr else result.stdout)

capstone_streamlit.py imports successfully.
Run:  streamlit run capstone_streamlit.py


## My Capstone Summary

**Name:** Siddharth Kumar Mishra

**Domain chosen:** Study Buddy — B.Tech Physics

**What the agent does:**
This agent is a 24/7 Physics study assistant for B.Tech students who need help at odd hours
when professors are unavailable. It explains physics concepts, laws, and derivations faithfully
from a 12-document knowledge base covering the full B.Tech syllabus — Newton's laws through
special relativity — without hallucinating formulas. It also has a built-in scientific calculator
that evaluates numerical expressions including trig, square roots, powers, and all key physics
constants (G, h, c, g, k, R, NA, mu0, eps0). All logic lives in `agent.py`; the notebook and
Streamlit UI both import from it so code is never duplicated.

**Knowledge base:**
12 documents, one topic each, 150-400 words:
Newton's Laws · Work/Energy/Power · Circular Motion & Gravitation ·
Waves & SHM · Thermodynamics · Electrostatics · Current Electricity ·
Magnetism & Electromagnetic Induction · Optics · Modern Physics ·
Special Relativity · Fluid Mechanics & Surface Tension.

**Tool used:**
Safe scientific calculator — `eval()` with a whitelisted namespace (no builtins).
Essential because students constantly need to substitute values into formulas
(e.g., Coulomb force, SHM period, free-fall velocity). The KB explains formulas
but cannot compute — the tool fills this gap. Never raises exceptions; returns
clear error strings so the graph never crashes.

**RAGAS baseline scores:**
- Faithfulness:       [run Part 6 to populate]
- Answer Relevance:   [run Part 6 to populate]
- Context Precision:  [run Part 6 to populate]

**Test results:** [run Part 5] / 10 tests passed. Red-team: [run Part 5] / 2 passed.

**One thing I would improve with more time:**
Replace the hand-written KB documents with actual B.Tech textbook PDFs parsed via
PyMuPDF, chunked at 300 tokens with 50-token overlap, and indexed with hybrid
BM25 + vector retrieval (`rank_bm25`). Hybrid search significantly improves context
precision for exact formula lookups where keyword matching outperforms dense vectors.

**Most surprising thing I learned building this:**
The router is the most fragile node — entirely because of prompt wording, not the LLM.
Writing "use tool for calculations" caused the router to send "what happens at 0 K?"
to tool (because it contains a number). Adding the word "explicitly" and listing
trigger keywords fixed it immediately. Prompt precision matters more than model size
for routing tasks.